# mAIME Evaluation Results Viewer

This notebook loads and displays results from lighteval mAIME evaluations.

In [7]:
import json
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

## Load Results

In [8]:
# Find the most recent results file
results_dir = Path("results_maime_full/results")

# Find all model directories
model_dirs = [d for d in results_dir.iterdir() if d.is_dir()]

if not model_dirs:
    print("No results found! Run the evaluation first.")
else:
    # Get the first model directory (or specify which one you want)
    model_dir = model_dirs[0]
    print(f"Loading results from: {model_dir.name}")
    
    # Get the most recent results file
    results_files = sorted(model_dir.glob("results_*.json"))
    if results_files:
        results_file = results_files[-1]
        print(f"Results file: {results_file.name}")
        
        with open(results_file) as f:
            results = json.load(f)
    else:
        print("No results files found!")

Loading results from: distilgpt2
Results file: results_2026-02-16T13-01-14.291865.json


## Summary Metrics

In [9]:
# Display overall results
print("=" * 60)
print(f"Model: {results['config_general']['model_name']}")
print(f"Evaluation time: {results['config_general']['total_evaluation_time_secondes']} seconds")
print(f"Max samples: {results['config_general']['max_samples']}")
print("=" * 60)
print("\nResults:")
print("-" * 60)

for task_name, metrics in results['results'].items():
    if task_name != "all":
        print(f"\n{task_name}:")
        for metric_name, value in metrics.items():
            print(f"  {metric_name}: {value:.4f}")

Model: distilgpt2
Evaluation time: 80.634663124918 seconds
Max samples: 5

Results:
------------------------------------------------------------

maime25:da|0:
  pass@k:k=1&n=1: 0.0000
  pass@k:k=1&n=1_stderr: 0.0000
  avg@n:n=1: 0.0000
  avg@n:n=1_stderr: 0.0000


## Task Configuration

In [10]:
# Show task configuration
for task_name, config in results['config_tasks'].items():
    print(f"\nTask: {task_name}")
    print(f"  Dataset: {config['hf_repo']} / {config['hf_subset']}")
    print(f"  Splits: {config['evaluation_splits']}")
    print(f"  Metrics: {[m['metric_name'] for m in config['metrics']]}")
    print(f"  Generation size: {config['generation_size']}")
    print(f"  Stop sequences: {config['stop_sequence']}")


Task: maime25:da|0
  Dataset: LumiOpen/mAIME2025 / da_combined
  Splits: ['test']
  Metrics: ['pass@k:k=1&n=1', 'avg@n:n=1']
  Generation size: None
  Stop sequences: []


## Load Detailed Results (if available)

In [14]:
# Load details parquet file if it exists
details_dir = Path("results_maime_full/details") / model_dir.name

if details_dir.exists():
    # Search recursively for parquet files (they're in timestamp subdirectories)
    details_files = sorted(details_dir.rglob("details_*.parquet"))
    if details_files:
        details_file = details_files[-1]
        print(f"Loading details from: {details_file.name}")
        print(f"Full path: {details_file}")
        
        df = pd.read_parquet(details_file)
        print(f"\nLoaded {len(df)} samples")
        print(f"\nColumns: {list(df.columns)}")
        
        # Show basic info
        correct_count = 0
        for _, row in df.iterrows():
            metric = row.get('metric', {})
            if metric.get('pass@k:k=1&n=1', 0.0) > 0:
                correct_count += 1
        print(f"\nCorrect answers: {correct_count} / {len(df)}")
    else:
        print("No details files found! Run with --save-details flag.")
        df = None
else:
    print("No details directory found! Run with --save-details flag.")
    df = None

Loading details from: details_maime25:da|0_2026-02-16T13-01-14.291865.parquet
Full path: results_maime_full/details/distilgpt2/2026-02-16T13-01-14.291865/details_maime25:da|0_2026-02-16T13-01-14.291865.parquet

Loaded 5 samples

Columns: ['doc', 'metric', 'model_response']


## View Individual Samples

In [17]:
if df is not None:
    # Function to display a sample
    def show_sample(idx):
        sample = df.iloc[idx]
        
        print("=" * 80)
        print(f"Sample {idx + 1}")
        print("=" * 80)
        
        # Extract data from nested structure
        doc = sample.get('doc', {})
        model_response = sample.get('model_response', {})
        metric = sample.get('metric', {})
        
        # Show prompt
        if 'query' in doc:
            print("\n📝 PROMPT:")
            print("-" * 80)
            prompt = doc['query']
            # Show first 500 chars
            print(prompt[:500] + "..." if len(prompt) > 500 else prompt)
        
        # Show model output
        if 'text' in model_response and len(model_response['text']) > 0:
            print("\n🤖 MODEL OUTPUT:")
            print("-" * 80)
            output = model_response['text'][0] if isinstance(model_response['text'], (list, tuple)) else model_response['text']
            # Show first 1000 chars
            print(output[:1000] + "..." if len(output) > 1000 else output)
            
            # Show truncation info
            if 'truncated_tokens_count' in model_response:
                print(f"\n⚠️  Truncated {model_response['truncated_tokens_count']} tokens")
        
        # Show gold answer
        if 'choices' in doc and len(doc['choices']) > 0:
            print("\n✅ GOLD ANSWER:")
            print("-" * 80)
            gold_idx = doc.get('gold_index', 0)
            print(doc['choices'][gold_idx])
        
        # Show extracted predictions and golds
        if 'specific' in doc:
            specific = doc['specific']
            if 'extracted_predictions' in specific:
                print("\n🔍 EXTRACTED PREDICTION:")
                print("-" * 80)
                print(specific['extracted_predictions'])
            if 'extracted_golds' in specific:
                print("\n🎯 EXTRACTED GOLD:")
                print("-" * 80)
                print(specific['extracted_golds'])
        
        # Show metrics
        print("\n📊 METRICS:")
        print("-" * 80)
        for metric_name, value in metric.items():
            print(f"{metric_name}: {value}")
    
    # Show first sample
    show_sample(0)
else:
    print("No detailed results available. Run with --save-details flag.")

Sample 1

📝 PROMPT:
--------------------------------------------------------------------------------

Solve the following math problem efficiently and clearly.  
The last line of your response should be of the following format: 
'Therefore, the final answer is: $\boxed{ANSWER}$. I hope it is correct' 
(without quotes) where ANSWER is just the final number or expression 
that solves the problem. Think step by step before answering.

Antag at $ \triangle ABC $ har vinklerne $ \angle BAC = 84^\circ $, $ \angle ABC = 60^\circ $ og $ \angle ACB = 36^\circ $. Lad $ D, E $ og $ F $ være midtpunkterne ...

🤖 MODEL OUTPUT:
--------------------------------------------------------------------------------
["The following math problem is simple:\n'Therefore, the final answer is: $\\boxed{ANSWER}$. I hope it is correct' \n(without quotes) where ANSWER is just the final number or expression \nthat solves the problem. Think step by step before answering.\nAntag at $ \\triangle ABC $ har vinklerne $ \\

## Browse All Samples

In [18]:
if df is not None:
    # Interactive: change this number to view different samples
    sample_number = 0  # Change this to view different samples (0 to len(df)-1)
    
    if 0 <= sample_number < len(df):
        show_sample(sample_number)
    else:
        print(f"Sample number must be between 0 and {len(df)-1}")
else:
    print("No detailed results available.")

Sample 1

📝 PROMPT:
--------------------------------------------------------------------------------

Solve the following math problem efficiently and clearly.  
The last line of your response should be of the following format: 
'Therefore, the final answer is: $\boxed{ANSWER}$. I hope it is correct' 
(without quotes) where ANSWER is just the final number or expression 
that solves the problem. Think step by step before answering.

Antag at $ \triangle ABC $ har vinklerne $ \angle BAC = 84^\circ $, $ \angle ABC = 60^\circ $ og $ \angle ACB = 36^\circ $. Lad $ D, E $ og $ F $ være midtpunkterne ...

🤖 MODEL OUTPUT:
--------------------------------------------------------------------------------
["The following math problem is simple:\n'Therefore, the final answer is: $\\boxed{ANSWER}$. I hope it is correct' \n(without quotes) where ANSWER is just the final number or expression \nthat solves the problem. Think step by step before answering.\nAntag at $ \\triangle ABC $ har vinklerne $ \\

## Summary Table

In [ ]:
if df is not None:
    # Create summary table
    summary_data = []
    for idx, row in df.iterrows():
        metric = row.get('metric', {})
        # Check pass@k metric
        passatk = metric.get('pass@k:k=1&n=1', 0.0)
        avgn = metric.get('avg@n:n=1', 0.0)
        
        summary_data.append({
            'sample_id': idx + 1,
            'correct': '✓' if passatk > 0 else '✗',
            'pass@k': passatk,
            'avg@n': avgn
        })
    
    summary_df = pd.DataFrame(summary_data)
    display(summary_df)
    
    # Show statistics
    correct_count = (summary_df['pass@k'] > 0).sum()
    accuracy = correct_count / len(df)
    print(f"\n📈 Accuracy: {accuracy:.2%} ({correct_count}/{len(df)})")
else:
    print("No detailed results available.")